In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2'
outdir = '/u/project/cluo/terencew/claude/igvf/pgp/ipsc_le_se_rerun/csv'
donors = ['C29', 'C37', 'C38', 'C39']
os.makedirs(outdir, exist_ok=True)

### iPSC-LE/SE assignments (3C Leiden clusters in snm3C, methylation liftover in snmCT)

In [ ]:
ips = pd.read_csv(f'{projdir}/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv', sep='\t', index_col=0)

In [ ]:
ips.head()

In [ ]:
ips.shape

### transcriptional-state labels: RNA (snmCT) and methylation-transferred (snm3C)

In [ ]:
rna = pd.read_csv(f'{projdir}/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv', sep='\t', index_col=0)
rna_qc = pd.read_csv(f'{projdir}/snmCT/rna/csv/label_transfer/snmct_xgboost_v7_filtered.csv', sep='\t', index_col=0)
m3c = pd.read_csv(f'{projdir}/snmCT/mc/csv/label_transfer/xgboost_time_v7.csv', sep='\t', index_col=0)
rna.shape, rna_qc.shape, m3c.shape

In [ ]:
ips['state_label'] = rna['pred_coarse'].reindex(ips.index)
mask = ips['assay'] == 'snm3C'
ips.loc[mask, 'state_label'] = m3c['cluster'].reindex(ips.index[mask])
ips['rna_qc'] = ips.index.isin(rna_qc.index)
ips['group'] = ips['ips_subtype'].replace({'IPS_main': 'iPSC-LE', 'IPS_alt': 'iPSC-SE'})
ips['early_ipsc'] = ips['state_label'] == 'IPS'
ips['time'] = ips['time'].astype(int)
ips[['assay', 'line', 'time', 'group', 'state_label', 'early_ipsc', 'rna_qc']].head()

In [ ]:
pd.crosstab([ips['assay'], ips['group']], ips['state_label'].fillna('no label'))

### retained counts per assay, group and donor

In [ ]:
counts = ips.groupby(['assay', 'group', 'line']).agg(n_all=('early_ipsc', 'size'), n_labelled=('state_label', lambda x: x.notna().sum()), n_early_ipsc=('early_ipsc', 'sum')).reset_index()
counts['n_removed'] = counts['n_all'] - counts['n_early_ipsc']
counts['pct_retained'] = (100 * counts['n_early_ipsc'] / counts['n_all']).round(1)
counts

In [ ]:
totals = counts.groupby(['assay', 'group'])[['n_all', 'n_labelled', 'n_early_ipsc', 'n_removed']].sum()
totals['pct_retained'] = (100 * totals['n_early_ipsc'] / totals['n_all']).round(1)
totals

### snmCT nuclei passing RNA QC (used in RNA panels)

In [ ]:
rna_counts = ips[(ips['assay'] == 'snmCT') & ips['rna_qc']].groupby(['group', 'line']).agg(n_all=('early_ipsc', 'size'), n_early_ipsc=('early_ipsc', 'sum')).reset_index()
rna_counts

### sampling day of retained nuclei

In [ ]:
pd.crosstab([ips.loc[ips['early_ipsc'], 'assay'], ips.loc[ips['early_ipsc'], 'group']], ips.loc[ips['early_ipsc'], 'time'])

### write

In [ ]:
ips.to_csv(f'{outdir}/ipsc_le_se_all_nuclei.csv', sep='\t')
ips[ips['early_ipsc']].to_csv(f'{outdir}/ipsc_le_se_early_ipsc_nuclei.csv', sep='\t')
counts.to_csv(f'{outdir}/counts_by_assay_group_donor.csv', sep='\t', index=False)
rna_counts.to_csv(f'{outdir}/counts_snmct_rna_qc_by_group_donor.csv', sep='\t', index=False)

In [ ]:
!date